In [1]:
import torch
from sklearn.model_selection import train_test_split
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
import random

from Circuits import Circuits
import GtoTmodel_09

In [2]:
circuits=Circuits()
graph_data,text_data= circuits.data_lodder()


Loading dataset files...


In [3]:
len(graph_data)

3349

In [4]:
# Combine graph_data and text_data into a single list of tuples
combined_data = list(zip(graph_data, text_data))

# Shuffle and split the data into train and test sets
train_data, test_data = train_test_split(combined_data, test_size=0.2, random_state=42)

# Unzip the train and test data back into separate lists
train_graph_data, train_text_data = zip(*train_data)
test_graph_data, test_text_data = zip(*test_data)

# Convert back to lists
train_graph_data, train_text_data = list(train_graph_data), list(train_text_data)
test_graph_data, test_text_data = list(test_graph_data), list(test_text_data)

print(f"Train size: {len(train_graph_data)}, Test size: {len(test_graph_data)}")

Train size: 2679, Test size: 670


In [ ]:
#Model parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embed_dim = 32  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 6 # Number of transformer layers
dropout = 0.1  # Dropout rate
graph_input_dim = 310  # Number of columns in the graph
text_vocab_size = 894  # Vocabulary size for text

model = GtoTmodel_09.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)

/home/nithira/circuits_gen/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [6]:
graph_dataset = torch.cat(graph_data).to(device)
text_dataset = torch.cat(text_data).to(device)
print(graph_dataset.shape, text_dataset.shape)

train_graph_dataset = torch.cat(train_graph_data).to(device)
train_text_dataset = torch.cat(train_text_data).to(device)
print(train_graph_dataset.shape, train_text_dataset.shape)

test_graph_dataset = torch.cat(test_graph_data).to(device)
test_text_dataset = torch.cat(test_text_data).to(device)
print(test_graph_dataset.shape, test_text_dataset.shape)

torch.Size([351461, 310]) torch.Size([351461])
torch.Size([283080, 310]) torch.Size([283080])
torch.Size([68381, 310]) torch.Size([68381])


In [7]:
def mask_graph_batch(batch, visibility):
    masked_batch = []
    for graph in batch:
        mask = (torch.rand_like(graph) < visibility).float()
        masked_graph = graph * mask
        masked_batch.append(masked_graph)
    return torch.nn.utils.rnn.pad_sequence(masked_batch, batch_first=True)


In [8]:
# Hyperparameters
learning_rate = 0.001
num_epochs = 100
batch_size = 128

# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Assuming 0 is the padding index
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Prepare data
graph_batches = [train_graph_data[i:i + batch_size] for i in range(0, len(train_graph_data), batch_size)]
text_batches = [train_text_data[i:i + batch_size] for i in range(0, len(train_text_data), batch_size)]

print(f"Number of batches: {len(graph_batches)}")

Number of batches: 21


In [ ]:
#Training Loop
import torch
import os

# Function to save the model every 10 epochs
def save_model(model, epoch, folder="GtoTSaves"):
    if not os.path.exists(folder):
        os.makedirs(folder)
    torch.save(model.state_dict(), os.path.join(folder, f"model_epoch_{epoch}.pth"))
    print(f"Model saved at epoch {epoch}")

# Training loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    # Compute visibility for current epoch
    initial_visibility = 0.5
    final_visibility = 0.2
    visibility = initial_visibility - (epoch / (num_epochs - 1)) * (initial_visibility - final_visibility)

    for graph_batch, text_batch in tqdm(zip(graph_batches, text_batches), total=len(graph_batches)):
        # Move text to device
        text_batch = torch.nn.utils.rnn.pad_sequence(text_batch, batch_first=True).to(device)
        # Apply masking to graphs and move to device
        graph_batch = mask_graph_batch(graph_batch, visibility).to(device)

        # Prepare input and target for the decoder
        decoder_input = text_batch[:, :-1]  # All except the last token
        decoder_target = text_batch[:, 1:]  # All except the first token

        # Forward pass
        outputs = model(graph_batch, decoder_input)

        # Compute loss
        outputs = outputs.reshape(-1, outputs.size(-1))  # Flatten for CrossEntropyLoss
        decoder_target = decoder_target.reshape(-1).long()  # Flatten target and cast to Long
        loss = criterion(outputs, decoder_target)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(graph_batches):.4f}")

    # Initialize test loss
    test_loss = 0
    test_graph_batches = [test_graph_data[i:i + batch_size] for i in range(0, len(test_graph_data), batch_size)]
    test_text_batches = [test_text_data[i:i + batch_size] for i in range(0, len(test_text_data), batch_size)]

    # Evaluation loop for test loss
    model.eval()
    with torch.no_grad():
        for graph_batch, text_batch in tqdm(zip(test_graph_batches, test_text_batches), total=len(test_graph_batches)):
            text_batch = torch.nn.utils.rnn.pad_sequence(text_batch, batch_first=True).to(device)
            graph_batch = torch.nn.utils.rnn.pad_sequence(graph_batch, batch_first=True).to(device)

            decoder_input = text_batch[:, :-1]
            decoder_target = text_batch[:, 1:]

            outputs = model(graph_batch, decoder_input)

            outputs = outputs.reshape(-1, outputs.size(-1))
            decoder_target = decoder_target.reshape(-1).long()
            loss = criterion(outputs, decoder_target)

            test_loss += loss.item()

    average_test_loss = test_loss / len(test_graph_batches)
    print(f"Test Loss: {average_test_loss:.4f}")

    # Save model every 10 epochs
    if (epoch + 1) % 10 == 0:
        save_model(model, epoch + 1)

print("Training complete.")


100%|██████████| 21/21 [00:04<00:00,  5.08it/s]


Epoch 1/100, Loss: 6.5312


100%|██████████| 6/6 [00:00<00:00, 22.66it/s]


Test Loss: 6.0270


100%|██████████| 21/21 [00:03<00:00,  5.37it/s]


...

Test Loss: 0.1594


100%|██████████| 21/21 [00:03<00:00,  5.54it/s]


Epoch 100/100, Loss: 0.1348


100%|██████████| 6/6 [00:00<00:00, 22.87it/s]

Test Loss: 0.1600
Model saved at epoch 100
Training complete.


In [ ]:
from sklearn.metrics import precision_score, accuracy_score
import numpy as np

# Final Evaluation
test_loss = 0
test_accuracy = 0
test_precision = 0
num_batches = len(test_graph_batches)

test_graph_batches = [test_graph_data[i:i + batch_size] for i in range(0, len(test_graph_data), batch_size)]
test_text_batches = [test_text_data[i:i + batch_size] for i in range(0, len(test_text_data), batch_size)]

model.eval()

# Open a file to save the output
with open("evaluation_output.txt", "w") as file:
    with torch.no_grad():
        all_preds = []
        all_targets = []

        file.write("Displaying the first 10 predictions and actual outputs:\n\n")

        # Counter to track the first 10 outputs
        count = 0

        for graph_batch, text_batch in tqdm(zip(test_graph_batches, test_text_batches), total=num_batches):
            text_batch = torch.nn.utils.rnn.pad_sequence(text_batch, batch_first=True).to(device)
            graph_batch = torch.nn.utils.rnn.pad_sequence(graph_batch, batch_first=True).to(device)

            decoder_input = text_batch[:, :-1]
            decoder_target = text_batch[:, 1:]

            outputs = model(graph_batch, decoder_input)

            # Calculate loss
            outputs = outputs.reshape(-1, outputs.size(-1))
            decoder_target = decoder_target.reshape(-1).long()
            loss = criterion(outputs, decoder_target)
            test_loss += loss.item()

            # Predicted values (take the highest probability class)
            _, predicted = torch.max(outputs, dim=-1)

            # Flatten the batch dimensions for accuracy and precision calculation
            all_preds.append(predicted.cpu().numpy())
            all_targets.append(decoder_target.cpu().numpy())

            # Print the first 10 predictions and actual values to the file
            if count < 1:
                file.write(f"Batch {count + 1}:\n")
                file.write("Predicted:  " + "\n")
                file.write(" ".join(map(str, predicted.cpu().numpy())) + "\n")
                file.write("Actual:  " + "\n")
                file.write(" ".join(map(str, decoder_target.cpu().numpy())) + "\n\n")
                count += 1

        # Calculate average loss
        average_test_loss = test_loss / num_batches
        file.write(f"Final Test Loss: {average_test_loss:.4f}\n")

        # Flatten lists of predictions and targets
        all_preds = np.concatenate(all_preds, axis=0)
        all_targets = np.concatenate(all_targets, axis=0)

        # Mask out padding tokens (assumed to be 0)
        mask = all_targets != 0
        all_preds = all_preds[mask]
        all_targets = all_targets[mask]

        # Calculate accuracy
        accuracy = accuracy_score(all_targets, all_preds)
        file.write(f"Test Accuracy: {accuracy * 100:.2f}%\n")

        # Calculate precision (micro average)
        precision = precision_score(all_targets, all_preds, average='micro')
        file.write(f"Test Precision: {precision:.4f}\n")


100%|██████████| 6/6 [00:00<00:00, 20.14it/s]


: 

In [ ]:
# Function to load the model from a saved file
def load_model(model, model_name, device='cpu'):
    model.load_state_dict(torch.load(model_name, map_location=device))
    model.to(device)  # Move model to the appropriate device (cpu or cuda)
    print(f"Model loaded from {model_name}")
    return model

# Example usage: Load the model from a specific file
model_name = "GtoTSaves/model_epoch_10.pth"  # Change this to the model file you want to load
model = load_model(model, model_name, device)


In [1]:
import GtoTmodel
device="cuda"
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 10 # Number of transformer layers
dropout = 0.1  # Dropout rate
graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 894  # Vocabulary size for text

model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)

c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [5]:
graph_dataset = torch.cat(graph_data).to(device)
text_dataset = torch.cat(text_data,).to(device)
graph_dataset.shape,text_dataset.shape

(torch.Size([351461, 310]), torch.Size([351461]))

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm

# Hyperparameters
learning_rate = 0.001
num_epochs = 50
batch_size = 100


# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Assuming 0 is the padding index
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Prepare data
graph_batches = [graph_data[i:i + batch_size] for i in range(0, len(graph_data), batch_size)]
text_batches = [text_data[i:i + batch_size] for i in range(0, len(text_data), batch_size)]

print(f"Number of batches: {len(graph_batches)}")

# Training loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for graph_batch, text_batch in tqdm(zip(graph_batches, text_batches), total=len(graph_batches)):
        # Move data to device
        
        text_batch = torch.nn.utils.rnn.pad_sequence(text_batch, batch_first=True).to(device)
        graph_batch = torch.nn.utils.rnn.pad_sequence(graph_batch, batch_first=True).to(device)

        # Prepare input and target for the decoder
        decoder_input = text_batch[:, :-1]  # All except the last token
        decoder_target = text_batch[:, 1:]  # All except the first token

        # Forward pass
        outputs = model(graph_batch, decoder_input)

        # Compute loss
        outputs = outputs.reshape(-1, outputs.size(-1))  # Flatten for CrossEntropyLoss
        decoder_target = decoder_target.reshape(-1).long()  # Flatten target and cast to Long
        loss = criterion(outputs, decoder_target)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(graph_batches):.4f}")

print("Training complete.")

Number of batches: 34


  0%|          | 0/34 [00:04<?, ?it/s]


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [6]:
# Define the file path to save the model and hyperparameters
save_path = "model_checkpoint.pth"

# Create a dictionary to store the model state and hyperparameters
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'embed_dim': embed_dim,
    'num_heads': num_heads,
    'num_layers': num_layers,
    'dropout': dropout,
    'learning_rate': learning_rate,
    'text_vocab_size': text_vocab_size,
    'graph_input_dim': graph_input_dim,
}

# Save the checkpoint
torch.save(checkpoint, save_path)
print(f"Model and hyperparameters saved to {save_path}")

Model and hyperparameters saved to model_checkpoint.pth


In [7]:
len(graph_data)

3349

In [8]:

# Initialize the model
model = TextToGraphTransformer(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout
)

NameError: name 'TextToGraphTransformer' is not defined

In [14]:
# Load the saved model checkpoint
checkpoint_path = "model_checkpoint.pth"
checkpoint = torch.load(checkpoint_path)

# Reinitialize the model with the saved hyperparameters
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim=checkpoint['graph_input_dim'],
    text_vocab_size=checkpoint['text_vocab_size'],
    embed_dim=checkpoint['embed_dim'],
    num_heads=checkpoint['num_heads'],
    num_layers=checkpoint['num_layers'],
    dropout=checkpoint['dropout']
).to(device)
model.eval()

# Select a sample graph input
sample_graph = graph_data[0].unsqueeze(0).to(device)  # Add batch dimension

# Generate a sequence
start_token = torch.tensor([2], dtype=torch.long).to(device)  # Assuming 0 is the start token
generated_sequence = [start_token.item()]

for _ in range(5000):  # Generate up to 50 tokens
    input_sequence = torch.tensor(generated_sequence, dtype=torch.long).unsqueeze(0).to(device)
    output = model(sample_graph, input_sequence)
    next_token = torch.argmax(output[:, -1, :], dim=-1).item()  # Get the most probable next token
    generated_sequence.append(next_token)
    if next_token == 892:  # Assuming 892 is the end token
        break

# Convert indices back to components
generated_text = circuits.get_component_fromlist(generated_sequence)

print("Generated Sequence:", generated_text)

C:\Users\MSI\AppData\Local\Temp\ipykernel_22020\247188143.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


KeyboardInterrupt: 